# Preprocessing loop

Generic preprocessing pipeline following the decisions in `../README.md`.

**How to use:**
1. Fill in `CONFIG` (paths and column lists). Optionally run the profiling cell first to help assign column types.
2. Run all cells. The result is `X_train, X_val, X_test, X_score` (plus `y_train, y_val, y_test`, `ids_score`).

Everything is fitted on the training split only; validation, test and scoring data are only transformed.

## 0. Environment (local)
Run in VS Code / Jupyter with the kernel **`C:\Users\migue\miniconda3\python.exe`**
(top-right "Select Kernel" → Python Environments → `miniconda3 (base)`).
The cell below checks that the required packages are available.

In [ ]:
import sys, importlib
print("Python:", sys.executable)
missing = []
for pkg in ["numpy", "pandas", "sklearn", "openpyxl", "xlrd", "msoffcrypto"]:
    # xlrd: old .xls files; msoffcrypto (pip: msoffcrypto-tool): password-protected files
    try:
        mod = importlib.import_module(pkg)
        print(f"  {pkg:9s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        missing.append(pkg)
if missing:
    pip_names = {"sklearn": "scikit-learn", "msoffcrypto": "msoffcrypto-tool"}
    names = [pip_names.get(p, p) for p in missing]
    raise ImportError(f"Missing packages: {missing}. Run in a terminal:\n"
                      f'  "{sys.executable}" -m pip install {" ".join(names)}')

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)

## 1. Configuration
Fill in the column lists. Columns not listed anywhere are ignored (a warning lists them).

In [ ]:
CONFIG = {
    # --- files ---
    # File names are looked up in data_dir (relative to this notebook's folder);
    # a full path (e.g. r"C:\...\DSC_training.xlsx") is used as-is.
    "data_dir": "../data",
    "train_path": "DSC_training.xlsx",
    "score_path": "DSC_scoring.xlsx",
    "sheet_name": 0,        # Excel sheet (index or name); ignored for .csv
    "use_cache": True,      # save a fast .pkl copy next to the Excel file; rebuilt when the Excel changes

    # --- splitting ---
    "val_size": 0.15,
    "test_size": 0.15,
    "random_state": 42,

    # --- missing values ---
    "max_missing_frac": 0.60,   # drop a column if more missing than this (on train)
    "add_missing_indicator": True,
    "missing_label": "Missing",

    # --- outliers / scaling (numeric) ---
    "clip_quantiles": (0.01, 0.99),   # None to disable
    "scaling": "zscore",              # "zscore", "minmax" or None

    # --- categorical ---
    "rare_min_frac": 0.01,     # levels below this share (train) -> "Other"
    "rare_label": "Other",
    "high_card_threshold": 40, # nominal with more distinct levels -> high-card encoding
    "high_card_method": "ratio",  # "ratio" (supervised ratio Vi) or "woe"; one-vs-rest per class
    "ratio_smoothing": 1.0,    # m-estimate weight toward the class prior (0 = none)
    "woe_adjustment": 0.5,     # added to counts to avoid log(0)
}

# --- columns: loaded from col_config.py (git-ignored; template: col_config.example.py) ---
import runpy
from pathlib import Path
_cols_file = Path("col_config.py")
if not _cols_file.is_file():
    raise FileNotFoundError("col_config.py not found: copy col_config.example.py "
                            "to col_config.py and fill in the column lists.")
CONFIG.update(runpy.run_path(str(_cols_file))["COLUMNS"])
for _k in ["id_cols", "numeric", "nominal", "high_card", "datetime", "drop"]:
    CONFIG.setdefault(_k, [])
CONFIG.setdefault("ordinal", {})
print("columns loaded from", _cols_file.resolve().name,
      {k: len(v) for k, v in CONFIG.items() if k in ["id_cols", "numeric", "nominal", "high_card", "ordinal", "datetime", "drop"]})


## 2. Load data

In [ ]:
PROJECT_DIR = Path.cwd()   # folder the notebook runs from (VS Code: the notebook's folder)
DATA_DIR = (PROJECT_DIR / CONFIG.get("data_dir", "../data")).resolve()

def find_file(name_or_path, data_dir=DATA_DIR):
    p = Path(name_or_path)
    candidates = [p] if p.is_absolute() else [data_dir / p, PROJECT_DIR / p]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(
        f"'{name_or_path}' not found. Looked in:\n"
        + "\n".join(f"  {c}" for c in candidates)
        + f"\nPut the data files in {data_dir}, or give a full path in CONFIG.")

import io
import getpass
import msoffcrypto

# In-memory store (survives re-running this cell, cleared on kernel restart).
_MEM = globals().setdefault("_MEM_TABLES", {})
_PWD = globals().setdefault("_FILE_PASSWORDS", {})

def _is_encrypted(path):
    if path.suffix.lower() == ".csv":
        return False
    with open(path, "rb") as f:
        try:
            return msoffcrypto.OfficeFile(f).is_encrypted()
        except Exception:
            return False   # not an OLE2 file (e.g. a normal .xlsx zip)

def _read_encrypted(path, sheet_name):
    # Decrypts into memory only - no decrypted copy is written to disk.
    key = str(path.parent)
    for attempt in range(3):
        if key not in _PWD:
            _PWD[key] = getpass.getpass(f"Password for {path.name}: ")
        buf = io.BytesIO()
        try:
            with open(path, "rb") as f:
                office = msoffcrypto.OfficeFile(f)
                office.load_key(password=_PWD[key])
                office.decrypt(buf)
            break
        except Exception:
            del _PWD[key]
            print("  wrong password (or unsupported protection), try again")
    else:
        raise PermissionError(f"Could not decrypt {path.name}. If it uses a sensitivity "
                              "label / IRM instead of a password, msoffcrypto cannot open it.")
    buf.seek(0)
    df = pd.read_excel(buf, sheet_name=sheet_name)
    buf.close()
    return df

def _read_source(path, sheet_name=0):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    try:
        if _is_encrypted(path):
            print(f"  {path.name} is password-protected: decrypting in memory")
            return _read_encrypted(path, sheet_name)
        return pd.read_excel(path, sheet_name=sheet_name)
    except PermissionError as e:
        if "decrypt" in str(e):
            raise
        raise PermissionError(
            f"Cannot open {path.name}: close it in Excel (Excel locks open files) and retry.") from e

def read_table(path, sheet_name=0, use_cache=True):
    # Reads csv/xlsx/xls (also password-protected Excel).
    # Normal files with use_cache: the first read saves '<file>.<sheet>.cache.pkl' next to the
    #   source; later runs load it in seconds; rebuilt when the source is newer.
    # Protected files: NEVER cached on disk; kept in memory for this kernel session only.
    mem_key = (str(path), sheet_name, path.stat().st_mtime)
    if mem_key in _MEM:
        print(f"  {path.name}: already in memory")
        return _MEM[mem_key].copy()
    if _is_encrypted(path):
        df = _read_source(path, sheet_name)
        _MEM[mem_key] = df
        return df.copy()
    if not use_cache:
        return _read_source(path, sheet_name)
    cache = path.with_name(f"{path.name}.{sheet_name}.cache.pkl")
    if cache.is_file() and cache.stat().st_mtime >= path.stat().st_mtime:
        print(f"  loading cache {cache.name}")
        return pd.read_pickle(cache)
    print(f"  reading {path.name} (slow for large Excel files, cached afterwards)...")
    df = _read_source(path, sheet_name)
    df.to_pickle(cache)
    return df

In [ ]:
train_file = find_file(CONFIG["train_path"])
score_file = find_file(CONFIG["score_path"])
print("training file:", train_file)
print("scoring file: ", score_file)

train_raw = read_table(train_file, CONFIG["sheet_name"], CONFIG["use_cache"])
score_raw = read_table(score_file, CONFIG["sheet_name"], CONFIG["use_cache"])
print("train:", train_raw.shape, " scoring:", score_raw.shape)

# early checks on the configuration
listed = (CONFIG["id_cols"] + CONFIG["numeric"] + CONFIG["nominal"] + CONFIG["high_card"]
          + list(CONFIG["ordinal"]) + CONFIG["datetime"] + CONFIG["drop"])
assert CONFIG["target"] in train_raw.columns, f"target '{CONFIG['target']}' not in training file"
unknown = [c for c in listed if c not in train_raw.columns]
assert not unknown, f"columns in CONFIG not found in training file: {unknown}"
dupes = sorted({c for c in listed if listed.count(c) > 1} - set(CONFIG["high_card"]))
if dupes:
    warnings.warn(f"columns listed in more than one CONFIG list: {dupes}")
missing_in_score = [c for c in listed if c not in score_raw.columns and c not in CONFIG["drop"]]
assert not missing_in_score, f"columns missing in scoring file: {missing_in_score}"
assert train_raw[CONFIG["target"]].notna().all(), "target has missing values"
print("class counts:", train_raw[CONFIG["target"]].value_counts().to_dict())

### (Optional) Profile column types
Heuristic guesses only, to help fill in `CONFIG`. Always confirm against the variable's meaning.

In [ ]:
def profile_types(df, cat_threshold=15):
    rows, n = [], len(df)
    for c in df.columns:
        s = df[c]
        nun = s.nunique(dropna=True)
        if nun <= 1:
            guess = "constant (drop?)"
        elif nun == 2:
            guess = "binary"
        elif pd.api.types.is_datetime64_any_dtype(s):
            guess = "datetime"
        elif pd.api.types.is_numeric_dtype(s):
            if nun >= 0.95 * n:
                guess = "ID or continuous - check"
            elif nun <= cat_threshold:
                guess = "categorical/ordinal coded as number? - check"
            else:
                guess = "numeric"
        else:
            if nun >= 0.95 * n:
                guess = "ID or free text"
            elif nun <= cat_threshold:
                guess = "nominal/ordinal categorical"
            else:
                guess = "high-cardinality categorical or text"
        rows.append((c, s.dtype, nun, round(s.isna().mean(), 3), guess))
    return pd.DataFrame(rows, columns=["col", "dtype", "n_unique", "frac_missing", "guess"])

# profile_types(train_raw)

In [ ]:
def suggest_config(df, target, cat_threshold=15, show_levels=8):
    n = len(df)
    groups = {k: [] for k in ["id_cols", "numeric", "nominal", "ordinal", "datetime", "drop"]}

    def add(group, col, note=""):
        groups[group].append((col, note))

    for c in df.columns:
        if c == target:
            continue
        s = df[c]
        nun = s.nunique(dropna=True)
        miss = s.isna().mean()
        miss_note = f"{miss:.0%} missing" if miss > 0 else ""

        if nun <= 1:
            add("drop", c, "constant")
            continue
        if pd.api.types.is_datetime64_any_dtype(s):
            add("datetime", c, miss_note)
            continue
        if pd.api.types.is_bool_dtype(s):
            add("numeric", c, "binary (bool)")
            continue
        if pd.api.types.is_numeric_dtype(s):
            if nun == n and pd.api.types.is_integer_dtype(s):
                add("id_cols", c, "CHECK: unique integer per row -> ID?")
            elif nun == 2:
                add("numeric", c, "binary 0/1")
            elif nun <= cat_threshold:
                add("numeric", c, f"CHECK: only {nun} values -> count, or code (nominal/ordinal)?")
            else:
                add("numeric", c, miss_note)
            continue

        # text / object columns: maybe numbers or dates stored as text?
        sample = s.dropna().astype(str).head(2000)
        if len(sample) and pd.to_numeric(sample.str.replace(",", ".", regex=False),
                                         errors="coerce").notna().mean() > 0.95:
            add("numeric", c, "CHECK: numbers stored as text (convert before use)")
            continue
        if len(sample) and sample.str.contains(r"\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4}", regex=True).mean() > 0.95:
            parsed = pd.to_datetime(sample, errors="coerce", dayfirst=True)
            if parsed.notna().mean() > 0.95:
                add("datetime", c, "CHECK: dates stored as text")
                continue
        if nun >= 0.95 * n:
            add("id_cols", c, "CHECK: almost unique text -> ID or free text?")
        elif nun == 2:
            add("nominal", c, "binary text")
        elif nun <= cat_threshold:
            levels = ", ".join(map(str, s.dropna().unique()[:show_levels]))
            add("nominal", c, f"CHECK: ordinal? levels: {levels}")
        else:
            add("nominal", c, f"{nun} levels" + (" -> high-card encoding" if nun > 40 else ""))

    # print a ready-to-paste block: replaces the lines from "target" to "drop" in CONFIG
        # print a ready-to-paste block: replaces the lines from "target" to "drop" in CONFIG
    import json
    q = lambda x: json.dumps(x, ensure_ascii=False)
    out = [f'    "target": {q(target)},']
    for g in ["id_cols", "numeric", "nominal", "high_card", "ordinal", "datetime", "drop"]:
        if g == "ordinal":
            out.append('    "ordinal": {},')
            continue
        cols = [c for c, _ in groups.get(g, [])]
        if not cols:
            out.append(f'    "{g}": [],')
            continue
        out.append(f'    "{g}": [')
        out += [f"        {q(c)}," for c in cols]
        out.append("    ],")
    print("\n".join(out))


   
suggest_config(train_raw, CONFIG["target"])


## 3. Train / validation / test split (stratified)

In [ ]:
def split_data(df, cfg):
    target = cfg["target"]
    X = df.drop(columns=[target])
    y = df[target]
    hold = cfg["val_size"] + cfg["test_size"]
    X_tr, X_hold, y_tr, y_hold = train_test_split(
        X, y, test_size=hold, stratify=y, random_state=cfg["random_state"])
    X_va, X_te, y_va, y_te = train_test_split(
        X_hold, y_hold, test_size=cfg["test_size"] / hold, stratify=y_hold,
        random_state=cfg["random_state"])
    return X_tr, X_va, X_te, y_tr, y_va, y_te

In [ ]:
X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = split_data(train_raw, CONFIG)
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s} n={len(y):6d}  class shares:", y.value_counts(normalize=True).round(3).to_dict())

## 4. Preprocessor

Steps, in order (all statistics learned in `fit` on the training split):

1. **Dates** → `year`, `month`, `weekday`, `is_weekend`; the raw date column is dropped. Derived columns are then treated as numeric.
2. **Drop columns** with more than `max_missing_frac` missing.
3. **Numeric**: clip at the 1st/99th percentile → median imputation (+ missing indicator) → scaling.
4. **Categorical** (nominal, high-card): missing → `"Missing"`; levels under `rare_min_frac` (and unseen levels) → `"Other"`.
   - Nominal with ≤ `high_card_threshold` levels → one-hot (all levels kept).
   - High-cardinality → one-vs-rest supervised ratio (Vi) or WoE: one column per class.
5. **Ordinal** → rank in the configured order; missing/unknown → -1.

In [ ]:
class Preprocessor:
    def __init__(self, cfg):
        self.cfg = cfg

    # ---------- helpers ----------
    @staticmethod
    def _date_features(X, cols):
        X = X.copy()
        new_cols = []
        for c in cols:
            d = pd.to_datetime(X[c], errors="coerce")
            X[f"{c}__year"] = d.dt.year
            X[f"{c}__month"] = d.dt.month
            X[f"{c}__weekday"] = d.dt.dayofweek
            X[f"{c}__is_weekend"] = (d.dt.dayofweek >= 5).astype(float).where(d.notna())
            new_cols += [f"{c}__year", f"{c}__month", f"{c}__weekday", f"{c}__is_weekend"]
        return X.drop(columns=cols), new_cols

    def _clean_cat(self, s, col):
        s = s.astype("object").where(s.notna(), self.cfg["missing_label"]).astype(str)
        return s.where(s.isin(self.levels_[col]), self.cfg["rare_label"])

    # ---------- fit ----------
    def fit(self, X, y):
        cfg = self.cfg
        X, date_cols = self._date_features(X, cfg["datetime"])

        listed = (set(cfg["numeric"]) | set(cfg["nominal"]) | set(cfg["high_card"])
                  | set(cfg["ordinal"]) | set(date_cols))
        ignored = set(X.columns) - listed - set(cfg["id_cols"]) - set(cfg["drop"])
        if ignored:
            warnings.warn(f"{len(ignored)} column(s) not in CONFIG are ignored: {sorted(ignored)}")

        # 2. drop columns with too many missings
        miss = X[[c for c in X.columns if c in listed]].isna().mean()
        self.dropped_missing_ = sorted(miss[miss > cfg["max_missing_frac"]].index)
        keep = lambda cols: [c for c in cols if c not in self.dropped_missing_]

        self.num_cols_ = keep(list(cfg["numeric"]) + date_cols)
        self.ord_cols_ = keep(list(cfg["ordinal"]))
        cat_cols = keep(list(dict.fromkeys(list(cfg["nominal"]) + list(cfg["high_card"]))))

        # 3. numeric
        Xn = X[self.num_cols_].apply(pd.to_numeric, errors="coerce")
        if cfg["clip_quantiles"] is not None:
            lo, hi = cfg["clip_quantiles"]
            self.clip_lo_, self.clip_hi_ = Xn.quantile(lo), Xn.quantile(hi)
            Xn = Xn.clip(self.clip_lo_, self.clip_hi_, axis=1)
        self.medians_ = Xn.median()
        self.indicator_cols_ = ([c for c in self.num_cols_ if Xn[c].isna().any()]
                                if cfg["add_missing_indicator"] else [])
        Xn = Xn.fillna(self.medians_)
        if cfg["scaling"] == "zscore":
            self.center_, self.scale_ = Xn.mean(), Xn.std(ddof=0)
        elif cfg["scaling"] == "minmax":
            self.center_, self.scale_ = Xn.min(), Xn.max() - Xn.min()
        elif cfg["scaling"] is not None:
            raise ValueError("scaling must be 'zscore', 'minmax' or None")
        if cfg["scaling"] is not None:
            self.scale_ = self.scale_.replace(0, 1.0)   # constant columns

        # 4. categorical: levels kept (rare -> Other), split low/high cardinality
        self.levels_, self.ohe_cols_, self.hc_cols_ = {}, [], []
        for c in cat_cols:
            s = X[c].astype("object").where(X[c].notna(), cfg["missing_label"]).astype(str)
            freq = s.value_counts(normalize=True)
            kept = set(freq[freq >= cfg["rare_min_frac"]].index) | {cfg["missing_label"]}
            self.levels_[c] = kept
            n_levels = s.nunique()
            if c in cfg["high_card"] or n_levels > cfg["high_card_threshold"]:
                self.hc_cols_.append(c)
            else:
                self.ohe_cols_.append(c)

        self.ohe_levels_ = {}
        for c in self.ohe_cols_:
            s = self._clean_cat(X[c], c)
            self.ohe_levels_[c] = sorted(s.unique())

        # high-cardinality: one-vs-rest supervised ratio / WoE per class
        y = pd.Series(np.asarray(y), index=X.index)
        self.classes_ = sorted(y.unique())
        self.hc_maps_, self.hc_default_ = {}, {}
        m, a = cfg["ratio_smoothing"], cfg["woe_adjustment"]
        for c in self.hc_cols_:
            s = self._clean_cat(X[c], c)
            counts = pd.crosstab(s, y).reindex(columns=self.classes_, fill_value=0)
            n_cat = counts.sum(axis=1)
            maps, defaults = {}, {}
            for k in self.classes_:
                pos = counts[k]
                if cfg["high_card_method"] == "ratio":
                    prior = (y == k).mean()
                    maps[k] = (pos + m * prior) / (n_cat + m)
                    defaults[k] = prior
                elif cfg["high_card_method"] == "woe":
                    neg = n_cat - pos
                    maps[k] = np.log(((pos + a) / (pos.sum() + a)) / ((neg + a) / (neg.sum() + a)))
                    defaults[k] = 0.0
                else:
                    raise ValueError("high_card_method must be 'ratio' or 'woe'")
            self.hc_maps_[c], self.hc_default_[c] = maps, defaults

        # 5. ordinal
        self.ord_maps_ = {c: {str(v): i for i, v in enumerate(cfg["ordinal"][c])}
                          for c in self.ord_cols_}

        self.feature_names_ = None
        self.feature_names_ = list(self.transform(X, _dates_done=True).columns)
        return self

    # ---------- transform ----------
    def transform(self, X, _dates_done=False):
        cfg = self.cfg
        if not _dates_done:
            X, _ = self._date_features(X, cfg["datetime"])
        parts = []

        # numeric
        if self.num_cols_:
            Xn = X[self.num_cols_].apply(pd.to_numeric, errors="coerce")
            if cfg["clip_quantiles"] is not None:
                Xn = Xn.clip(self.clip_lo_, self.clip_hi_, axis=1)
            ind = Xn[self.indicator_cols_].isna().astype(int).add_suffix("__missing")
            Xn = Xn.fillna(self.medians_)
            if cfg["scaling"] is not None:
                Xn = (Xn - self.center_) / self.scale_
            parts += [Xn, ind]

        # one-hot
        for c in self.ohe_cols_:
            s = self._clean_cat(X[c], c)
            parts.append(pd.DataFrame({f"{c}__{lv}": (s == lv).astype(int)
                                       for lv in self.ohe_levels_[c]}, index=X.index))

        # high-cardinality
        tag = "ratio" if cfg["high_card_method"] == "ratio" else "woe"
        for c in self.hc_cols_:
            s = self._clean_cat(X[c], c)
            parts.append(pd.DataFrame(
                {f"{c}__{tag}_{k}": s.map(self.hc_maps_[c][k]).astype(float)
                                     .fillna(self.hc_default_[c][k])
                 for k in self.classes_}, index=X.index))

        # ordinal
        if self.ord_cols_:
            parts.append(pd.DataFrame(
                {c: X[c].astype(str).map(self.ord_maps_[c]).fillna(-1).astype(int)
                 for c in self.ord_cols_}, index=X.index))

        out = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=X.index)
        if self.feature_names_ is not None:
            out = out.reindex(columns=self.feature_names_, fill_value=0)
        return out

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

    def summary(self):
        return pd.Series({
            "dropped (too many missing)": len(self.dropped_missing_),
            "numeric (incl. date parts)": len(self.num_cols_),
            "missing indicators": len(self.indicator_cols_),
            "one-hot variables": len(self.ohe_cols_),
            "high-cardinality variables": len(self.hc_cols_),
            "ordinal variables": len(self.ord_cols_),
            "output features": len(self.feature_names_),
        })

## 5. Fit on train, transform everything

In [ ]:
pre = Preprocessor(CONFIG).fit(X_train_raw, y_train)

X_train = pre.transform(X_train_raw)
X_val   = pre.transform(X_val_raw)
X_test  = pre.transform(X_test_raw)

ids_score = score_raw[CONFIG["id_cols"]].copy() if CONFIG["id_cols"] else None
X_score = pre.transform(score_raw.drop(columns=[CONFIG["target"]], errors="ignore"))

print(pre.summary().to_string())
print("dropped for missingness:", pre.dropped_missing_)

## 6. Sanity checks

In [ ]:
for name, df in [("train", X_train), ("val", X_val), ("test", X_test), ("score", X_score)]:
    assert list(df.columns) == pre.feature_names_, f"{name}: column mismatch"
    assert not df.isna().any().any(), f"{name}: contains NaN"
    print(f"{name:5s} shape={df.shape}")